<a href="https://colab.research.google.com/github/Sunil-Paliwal/ai-enabled-drug-discovery-and-synthesis/blob/main/04_aizynthfinder_retrosynthesis/build_stock_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Build a stock file for AiZynthFinder

Turns a supplier catalogue (Enamine, ZINC, anyone's) into the stock file
AiZynthFinder uses to decide when a molecule is buyable and the search can stop.

**Author:** Dr. Sunil Paliwal
Department of Chemistry and Chemical Biology
Stevens Institute of Technology

---

**What comes out.** A single `.hdf5` holding one column of InChIKeys &mdash; nothing
else. An InChIKey is a hash: it answers *"is this molecule in the set?"* and nothing
more. No structures, no catalogue numbers, no prices, and the catalogue cannot be
reconstructed from it. Worth saying to a supplier when asking whether you may share
the result.

**Why bother.** AiZynthFinder's default stock is a snapshot of ZINC from April 2020.
"Was somewhere in ZINC six years ago" is a weak stopping criterion. "Is in our
supplier's current catalogue" is a real one.

**Run the cells in order.** The conversion is the slow part &mdash; roughly 1,000&ndash;4,000
structures a second, so a million-compound catalogue takes a few minutes.

---
### Step 1 &mdash; Install

In [ ]:
!pip install rdkit pandas tables --quiet
print("ready")

---
### Step 2 &mdash; Point at your catalogue

**Compress the SDF before you move it.** A multi-gigabyte catalogue shrinks to
roughly a tenth of its size, and this notebook reads the compressed file
directly &mdash; there is no need to unpack it. On Windows, right-click the file
and choose *Send to &rarr; Compressed (zipped) folder*; on macOS, right-click and
choose *Compress*. Either way you get a `.zip`, which is fine.

**Then put it in Google Drive** and run this cell. Drive uploads survive a
dropped connection and you only do them once; Colab's own upload widget is
unreliable past about a gigabyte, and anything uploaded to `/content` is lost
when the session ends, so a timeout means starting over.

Set `CATALOGUE` to the path it lands at &mdash; something like
`/content/drive/MyDrive/Enamine_BB.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# have a look around to find your file:
!ls -la /content/drive/MyDrive/ | head -30

---
### Step 3 &mdash; Set the paths

The only cell you edit. Accepts `.sdf`, `.sdf.gz`, `.zip`, or a plain text file
of SMILES (one per line).

Give `OUTPUT` a name of its own for each catalogue &mdash; writing a second one to
the same filename overwrites the first.

In [ ]:
CATALOGUE = "/content/drive/MyDrive/Enamine_BB.sdf"   # <-- your file
OUTPUT    = "/content/enamine_bb_stock.hdf5"          # <-- what to write
SOURCE    = "enamine"                                 # <-- a short label

import os
print("found it" if os.path.exists(CATALOGUE) else "NOT FOUND - check the path")
if os.path.exists(CATALOGUE):
    print(f"{os.path.getsize(CATALOGUE)/1e6:,.0f} MB")

---
### Step 4 &mdash; Convert

Streams the file rather than loading it, so memory stays flat however big the
catalogue is. Progress prints every 100,000 structures. A few unreadable entries are
normal &mdash; every supplier catalogue has salts and oddities RDKit will not parse.

In [ ]:
import gzip, zipfile, os, time, warnings
warnings.filterwarnings("ignore")
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

_CHEM_EXT = (".sdf", ".mol", ".sd", ".smi", ".ism", ".txt")


def _zip_members(path):
    """Every chemistry file inside a zip, largest first.

    A supplier archive often holds the catalogue split across several SDFs,
    sometimes alongside a small sample or readme. Reading only the first
    entry silently converts the wrong file, so read them all -- and report
    what was found, so a surprise is visible rather than silent.
    """
    zf = zipfile.ZipFile(path)
    members = [i for i in zf.infolist()
               if not i.is_dir()
               and i.filename.lower().endswith(_CHEM_EXT)
               and not i.filename.startswith("__MACOSX")]
    if not members:
        raise ValueError(
            f"No SDF or SMILES file inside {path}.\n"
            f"The archive holds: {[i.filename for i in zf.infolist()[:10]]}")
    members.sort(key=lambda i: -i.file_size)
    print(f"  {len(members)} chemistry file(s) in the archive:")
    for i in members:
        print(f"    {i.file_size/1e6:>10,.1f} MB uncompressed   {i.filename}")
    return zf, members


def _iter_stream(fh, name):
    if name.lower().endswith((".sdf", ".mol", ".sd")):
        for mol in Chem.ForwardSDMolSupplier(fh, sanitize=True, removeHs=True):
            yield mol
    else:
        for raw in fh:
            line = raw.decode("utf-8", "replace").strip()
            if not line or line.startswith("#"):
                continue
            token = line.split()[0]
            if token.lower() in ("smiles", "smi"):      # header row
                continue
            yield Chem.MolFromSmiles(token)


def iter_mols(path):
    """Yield RDKit mols from an SDF or a file of SMILES.

    Accepts .sdf, .sdf.gz, .zip (including multi-file archives), .smi, .txt.
    """
    low = path.lower()
    if low.endswith(".zip"):
        zf, members = _zip_members(path)
        for info in members:
            with zf.open(info) as fh:
                yield from _iter_stream(fh, info.filename)
        zf.close()
        return
    fh = gzip.open(path, "rb") if low.endswith(".gz") else open(path, "rb")
    inner = path[:-3] if low.endswith(".gz") else path
    try:
        yield from _iter_stream(fh, inner)
    finally:
        fh.close()

keys, read, failed = set(), 0, 0
t0 = time.time()
for mol in iter_mols(CATALOGUE):
    read += 1
    if mol is None:
        failed += 1
    else:
        try:
            keys.add(Chem.MolToInchiKey(mol))
        except Exception:
            failed += 1
    if read % 100_000 == 0:
        print(f"  {read:>10,} read   {len(keys):>10,} unique   "
              f"{read/max(time.time()-t0,1e-9):>7,.0f}/s")

keys.discard("")
pd.DataFrame({"inchi_key": sorted(keys)}).to_hdf(OUTPUT, key="table", mode="w")

print(f"\nDone in {(time.time()-t0)/60:.1f} min")
print(f"  structures read   {read:,}")
print(f"  unique InChIKeys  {len(keys):,}")
print(f"  unreadable        {failed:,}")
print(f"\nWrote {OUTPUT}  ({os.path.getsize(OUTPUT)/1e6:,.1f} MB)")

if len(keys) < 10_000:
    print("\n*** WARNING ***")
    print(f"Only {len(keys):,} unique structures. A supplier catalogue should run")
    print("to hundreds of thousands. Check the file listing printed above: the")
    print("converter may have read a sample file rather than the catalogue, or")
    print("the archive may not contain what you think it does.")

---
### Step 5 &mdash; Check it, then download

Confirms the file is in the format AiZynthFinder reads, and spot-checks a lookup.

In [ ]:
import pandas as pd
from rdkit import Chem

df = pd.read_hdf(OUTPUT, "table")
print(f"{len(df):,} InChIKeys, column = {list(df.columns)}")

lookup = set(df["inchi_key"])
for smi in ["CC(=O)O", "OB(O)c1ccc(F)cc1", "c1ccc2cc3ccccc3cc2c1"]:
    k = Chem.MolToInchiKey(Chem.MolFromSmiles(smi))
    print(f"  {smi:24s} in stock: {k in lookup}")

from google.colab import files
files.download(OUTPUT)

---
### Using it

Put the file beside AiZynthFinder's config and name it under `stock`:

```yaml
stock:
  enamine: enamine_bb_stock.hdf5
  zinc:    zinc_stock.hdf5
```

List more than one and AiZynthFinder reports which stock each starting material came
from &mdash; so a route can show that one building block is current Enamine catalogue
while another only appears in the 2020 ZINC snapshot. That distinction is worth
showing a student: *commercially available* is not one thing.

---

<div style="border:2px solid #3f7d58; border-radius:10px; padding:4px 20px 14px 20px;
            background:#f4faf6; color:#1b1b1b; font-size:14px;">

<h3 style="color:#2c5c40; margin-bottom:2px;"><b>What this does not tell you</b></h3>

<p>A stock file answers one question: is this exact structure in the catalogue. It
knows nothing about price, pack size, lead time, or whether the supplier actually has
it on the shelf today. A route that stops at a &quot;buyable&quot; building block may
still stop at something that costs a fortune or ships in eight weeks.</p>

<p>Stereochemistry matters here. InChIKey lookup is exact, so a specific enantiomer
will not match a racemate in the catalogue, and vice versa. If routes are failing at
starting materials you believe are purchasable, this is the first thing to check.</p>

</div>

<br>